# Concatenating and merging data

Functions to concatenate or merge DataFrames in pandas:

1. [`pd.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html): combine multiple DataFrames by appending observations (rows) or columns
2. [`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html): match observations from one DataFrame with observations from another DataFrame

You can also consult the official 
[user guide](https://pandas.pydata.org/docs/user_guide/merging.html) 
and the pandas 
[cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) 
for more information.

***
## Concatenation

### Concatenating Series

*Example: Concatenating two Series along the row axis*

- Specify `pd.concat(..., axis=0)` (or use default value)

*Example: Concatenating along the column axis*

- Specify `pd.concat(..., axis=1)`
- Usually only makes sense for equal-length `Series`
- Explicitly set columns names using `pd.concat(..., keys=...)`

<div class="alert alert-info">
<h3> Your turn</h3>
<ol>
    <li>Create a new <TT>Series</TT> with observations <TT>['C1', 'C2']</TT>.</li>
    <li>Using the previously created <TT>Series</TT> <TT>a</TT> and <TT>b</TT>, concatenate all three objects along the row axis and create a new (unique) index.</li>
    <li>Repeat the previous step, but now concatenate along the column axis. Assign the column names <TT>'Column1'</TT>, <TT>'Column2'</TT>, and <TT>'Column3'</TT>.</li>
</ol>
</div>

***
### Concatenating DataFrames

#### Concatenating along the column axis

*Example: Concatenating two DataFrames along the column axis*

- Specify `pd.concat(..., axis=1)`

*Example: Concatenating a DataFrame and a Series*

#### Concatenating along the row axis

*Example: Concatenating rows with identical columns*

*Example: Concatenating rows with different columns*

- Creates `NaN`s for non-overlapping columns

<div class="alert alert-info">
<h3> Your turn</h3>
Use the data files located in the folder <TT>../data/FRED</TT> to perform the following tasks:
<ol>
    <li>Load the data in <TT>FRED_monthly_1950.csv</TT> and <TT>FRED_monthly_1960.csv</TT> into two different DataFrames.
        The files contain monthly macroeconomic time series for the 1950s and 1960s, respectively.
        <br/>
        <i>Hint:</i> Use <TT>pd.read_csv(..., parse_dates=['DATE'])</TT> to automatically parse strings stored in the <TT>DATE</TT> column as dates.
        </li>
    <li>Concatenate these DataFrames along the row dimension to get a total of 240 observations.</li>
    <li>Set the column <TT>Date</TT> as index for the newly created DataFrame.</li>
</ol>
</div>

In [80]:
import pandas as pd
import numpy as np

DATA_PATH = '/Users/andreas/NHH/Tech2-local/TECH2-H24/data/FRED'
df1 = pd.read_csv(f'{DATA_PATH}/FRED_monthly_1950.csv', parse_dates=['DATE'])
df2 = pd.read_csv(f'{DATA_PATH}/FRED_monthly_1960.csv', parse_dates=['DATE'])
df1

,DATE,CPI,UNRATE,FEDFUNDS,REALRATE,LFPART
0,1950-01-01,23.5,6.5,NaN,NaN,58.9
1,1950-02-01,23.6,6.4,NaN,NaN,58.9
2,1950-03-01,23.6,6.3,NaN,NaN,58.8
3,1950-04-01,23.6,5.8,NaN,NaN,59.2
4,1950-05-01,23.8,5.5,NaN,NaN,59.1
...,...,...,...,...,...,...
115,1959-08-01,29.2,5.2,3.5,NaN,59.2
116,1959-09-01,29.2,5.5,3.8,NaN,59.3
117,1959-10-01,29.4,5.7,4.0,NaN,59.4
118,1959-11-01,29.4,5.8,4.0,NaN,59.1


In [81]:
df = pd.concat((df1, df2)).reset_index(drop=True)
df.set_index(['DATE'], inplace =True)
df

,CPI,UNRATE,FEDFUNDS,REALRATE,LFPART
DATE,,,,,
1950-01-01,23.5,6.5,NaN,NaN,58.9
1950-02-01,23.6,6.4,NaN,NaN,58.9
1950-03-01,23.6,6.3,NaN,NaN,58.8
1950-04-01,23.6,5.8,NaN,NaN,59.2
1950-05-01,23.8,5.5,NaN,NaN,59.1
...,...,...,...,...,...
1969-08-01,36.9,3.5,9.2,NaN,60.3
1969-09-01,37.1,3.7,9.2,NaN,60.3
1969-10-01,37.3,3.7,9.0,NaN,60.4


***
## Merging and joining data sets

### Types of merges

We can merge DataFrames `A` and `B` in various ways:

1.  *one-to-one*: Unique keys in `A` and `B`
2.  *many-to-one*: Unique keys in `A`, non-unique keys in `B`
3.  *many-to-many*: Non-unique keys in both `A` and `B` (avoid this)

### Implementation in pandas

1.  [`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html): function that takes as argument the *two* DataFrames to be merged,
    e.g.,
    
    ```python
    result = pd.merge(df_A, df_B)
    ```
2.  [`df.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html): method of a specific `DataFrame` object, takes the other `DataFrame` as argument, e.g.,
    
    ```python
    result = df_A.merge(df_B)
    ```


### Controlling the resulting data set

Merge methods:

1.  `how='inner'` (_inner join_): _intersection_ of keys that are present in _both_ data sets
2.  `how='outer'` (_outer join_): _union_ of keys present in either of the data sets
3.  `how='left'` (_left join_): all identifiers from the _left_ data set are in the result
4.  `how='right'` (_right join_): all identifiers from the _right_ data set are in the result

Illustration: Each circle presents the keys present in the left (`df1`) or right (`df2`) DataFrames. The merge method controls which subset of keys is retained in the merge result.

![Join types](../lecture4/join-methods.png)

***
### Merging with `merge()`

- Create two DataFrames with partially overlapping keys

In [82]:
df_a = pd.DataFrame({'key': [0,1], 'value_a': ['A0','A1']})
df_a

,key,value_a
0,0,A0
1,1,A1


In [83]:
df_b = pd.DataFrame({'key': [1,2], 'value_b': ['B1','B2']})
df_b 

,key,value_b
0,1,B1
1,2,B2


#### Using `pd.merge()`

In [84]:
df = pd.merge(df_a, df_b, on='key', how='right')
df

,key,value_a,value_b
0,1,A1,B1
1,2,NaN,B2


*Example: one-to-one merges*

Merge using the following methods:

1.  `inner` (default for `pd.merge()`)
2.  `outer`
3.  `left`
4.  `right`

#### Using `DataFrame.merge()`

*Example: Merging with overlapping column names*

- pandas automatically renames overlapping column names not used as keys
- Can use `merge(..., suffixes=...)` to override default suffixes for _left_ and _right_

<div class="alert alert-info">
<h3> Your turn</h3>
Use the data files located in the folder <TT>../data/FRED</TT> to perform the following tasks:
<ol>
    <li>Load the data in <TT>CPI.csv</TT> and <TT>GDP.csv</TT> into two different DataFrames.
        The files contain monthly data for the Consumer Price Index (CPI) and quarterly data for GDP, respectively.
        <br/>
        <i>Hint:</i> Use <TT>pd.read_csv(..., parse_dates=['DATE'])</TT> to automatically parse strings stored in the <TT>DATE</TT> column as dates.
        </li>
    <li>Merge the CPI with the GDP time series with 
    <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html"><TT>merge()</TT></a> 
    using a left join (<TT>how='left'</TT>). How many observations does the resulting DataFrame have?</li>
    <li>Merge the CPI with the GDP time series with <TT>merge()</TT> using an inner join (<TT>how='inner'</TT>). How many observations does the resulting DataFrame have,
        and why is this different from the previous case?</li>
</ol>
</div>

In [85]:
df1 = pd.read_csv(f'{DATA_PATH}/CPI.csv',parse_dates=['DATE'])
df2 = pd.read_csv(f'{DATA_PATH}/GDP.csv',parse_dates=['DATE'])

df = pd.merge(df1, df2, on= 'DATE', how ='left').set_index('DATE', inplace=True)
df

In [86]:
df = pd.merge(df1, df2, on= 'DATE', how ='inner')
df.set_index('DATE', inplace=True)
df

,CPI,GDP
DATE,,
1947-01-01,21.5,2182.7
1947-04-01,22.0,2176.9
1947-07-01,22.2,2172.4
1947-10-01,22.9,2206.5
1948-01-01,23.7,2239.7
...,...,...
2023-04-01,303.0,22539.4
2023-07-01,304.6,22780.9
2023-10-01,307.5,22960.6


***
### Joining with `join()`

The `DataFrame` method 
[`join()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) 
is a convenience wrapper around 
[`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html):

1.  `join()` can be called _only_ directly on the `DataFrame` object
2.  `join()` always operates on the _index_ of the other `DataFrame`
3.  `join()` by default performs a `left` join, whereas `merge()` performs an `inner` join

Use `join()` if you want to join DataFrames which have a similar index.

*Example: joining DataFrames*

- Create two demo DataFrames with keys as index
- `join()` with `how='left'` and `how='inner'`

In [87]:
df_a = df_a.set_index('key')
df_b = df_b.set_index('key')

In [88]:
df_a.join(df_b)

,value_a,value_b
key,,
0,A0,NaN
1,A1,B1


<div class="alert alert-info">
<h3> Your turn</h3>
Use the data files located in the folder <TT>../data/FRED</TT> to perform the following tasks:
<ol>
    <li>Load the data in <TT>CPI.csv</TT> and <TT>GDP.csv</TT> into two different DataFrames.
        The files contain monthly data for the Consumer Price Index (CPI) and quarterly data for GDP, respectively.
        <br/>
        <i>Hint:</i> Use <TT>pd.read_csv(..., parse_dates=['DATE'])</TT> to automatically parse strings stored in the <TT>DATE</TT> column as dates.
        </li>
    <li>Set the <TT>DATE</TT> column as the index for each of the two DataFrames.</li>
    <li>Merge the CPI with the GDP time series with 
    <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html"><TT>join()</TT></a>. 
    Do this with both a left and an inner join.</li>
</ol>
</div>

In [89]:
df1 = pd.read_csv(f'{DATA_PATH}/CPI.csv',parse_dates=['DATE']).set_index('DATE')
df2 = pd.read_csv(f'{DATA_PATH}/GDP.csv',parse_dates=['DATE']).set_index('DATE')

df1.join(df2, how = 'left')

,CPI,GDP
DATE,,
1947-01-01,21.5,2182.7
1947-02-01,21.6,NaN
1947-03-01,22.0,NaN
1947-04-01,22.0,2176.9
1947-05-01,22.0,NaN
...,...,...
2024-04-01,313.2,23223.9
2024-05-01,313.2,NaN
2024-06-01,313.0,NaN


In [90]:
df1.join(df2, how = 'right')

,CPI,GDP
DATE,,
1947-01-01,21.5,2182.7
1947-04-01,22.0,2176.9
1947-07-01,22.2,2172.4
1947-10-01,22.9,2206.5
1948-01-01,23.7,2239.7
...,...,...
2023-04-01,303.0,22539.4
2023-07-01,304.6,22780.9
2023-10-01,307.5,22960.6


***
# Dealing with missing values

- Often arise if concatenated / merged data sets don't contain the same columns or indices

## Dropping missing values

Missing values can be dropped by either

1. Using [`dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)
    or selecting a subset of observations with a boolean operation such as 
    [`notna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html).
2. Avoiding the missing values in the first place, e.g., by using `merge(..., how='inner')`.

*Example: Dropping missing values*

- Use _outer/left/right join_ and drop missing data with `dropna()`

*Example: Avoiding missing values in the first place*

- E.g., use _inner join_ when merging data sets

## Filling missing values

Instead of dropping data, we can impute missing values in various ways:

1.  [`fillna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html): replace missing data with user-specified values
2.  [`ffill()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ffill.html) and 
    [`bfill()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.bfill.html): fill missing values
    forward or backward from adjacent non-missing observations
3.  [`interpolate()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.interpolate.html): interpolate missing values based on non-missing ones

*Example: Replacing missing values with `fillna()`*

- Specify one value for _entire_ `DataFrame`
- Specify different value for each column

*Example: forward- or backward-filling missing values*

- Use [`ffill()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ffill.html) to forward-fill
- Use [`bfill()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.bfill.html) to backward-fill

*Example: linear interpolation*

- Use [`interpolate()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.interpolate.html) to interpolate missing _numerical_ data

<div class="alert alert-info">
<h3> Your turn</h3>
Use the data files located in the folder <TT>../data/FRED</TT> to perform the following tasks:
<ol>
    <li>Load the data in <TT>CPI.csv</TT> and <TT>GDP.csv</TT> into two different DataFrames.
        The files contain monthly data for the Consumer Price Index (CPI) and quarterly data for GDP, respectively.
        <br/>
        <i>Hint:</i> Use <TT>pd.read_csv(..., parse_dates=['DATE'])</TT> to automatically parse strings stored in the <TT>DATE</TT> column as dates.
        </li>
    <li>Merge the CPI with the GDP time series with <TT>merge()</TT> using a left join. This creates missing values in the <TT>GDP</TT>
    column.</li>
    <li>Impute the missing GDP values using <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.interpolate.html"><TT>interpolate()</TT></a> 
    and replace the missing values in column <TT>GDP</TT>.</li>
</ol>
</div>

In [101]:
df1.join(df2, how='left').interpolate(method='linear').round(2)

,CPI,GDP
DATE,,
1947-01-01,21.5,2182.70
1947-02-01,21.6,2180.77
1947-03-01,22.0,2178.83
1947-04-01,22.0,2176.90
1947-05-01,22.0,2175.40
...,...,...
2024-04-01,313.2,23223.90
2024-05-01,313.2,23223.90
2024-06-01,313.0,23223.90
